# 02 — PhoWhisper-small on FLEURS `vi_vn`: zero-shot, LoRA fine-tuning, evaluation

Self-contained. Before running: *Settings → Accelerator: GPU T4*, *Internet: on*, and
*Add Input* → a FLEURS dataset saved with Hugging Face `save_to_disk` that has a `vi_vn`
folder (e.g. `fleurs-asian-6lang`).

| Step | What | Output in `/kaggle/working` |
|---|---|---|
| 0 | Configuration | |
| 1 | Clone the repo, install it (only audio, text and scoring helpers are used) | |
| 2 | Load FLEURS `vi_vn`, build the test / dev tables (clean + optional noise) | `test_set.csv`, `dev_set.csv` |
| 3 | Load PhoWhisper-small, `transcribe(model, processor, audios)`, shared helpers | |
| 4 | Sanity check on 10 files, then the full zero-shot test | `predictions_phowhisper-small_zero-shot.csv` |
| 5 | Evaluate: WER / CER per condition, bootstrap intervals | |
| 6 | Fine-tuning setup: training set, collate, LoRA | |
| 7 | Fine-tuning, dev WER after each epoch, keep the best adapter | `lora_*/`, `history_*.csv` |
| 8 | Test the best adapter, compare with zero-shot | `predictions_phowhisper-small_<variant>.csv` |

**Noise.** With `NOISE_DIRS = {}` everything is clean. To add noise, attach folders of noise
recordings and list them, e.g. `{"traffic": "/kaggle/input/<x>/traffic"}`: every test
utterance is then also mixed with each noise type at each SNR in `SNRS`, and fine-tuning
mixes noise on the fly (variant `ft-noisy`). Noise files are split by file into test and
train, so test noise is never heard in training.

Steps 6–8 can run in a later session: run steps 0–3, then jump to 6.

## 0. Configuration

In [ ]:
from pathlib import Path

# Code (only core / text / scoring helpers are used)
REPO_URL = "https://github.com/anhvinhphan659/asr-noisy-vi"
COMMIT = "83fad8741d1cb742f26250a1f23dd6880c764b60"
CODE_DIR = Path("/tmp/asr-noisy-vi")

# FLEURS (Hugging Face save_to_disk layout: <root>/vi_vn/dataset_dict.json)
FLEURS_DIR = None            # None = search /kaggle/input for vi_vn/dataset_dict.json
LANG = "vi_vn"
SPLITS = {"train": "train", "dev": "validation", "test": "test"}
MAX_DURATION_S = 30.0        # Whisper reads at most 30 s
LIMIT_TEST = None            # utterances (None = all); e.g. 20 for a dry run
LIMIT_DEV = None

# Noise (optional): {noise_type: folder of recordings}; {} = clean only
NOISE_DIRS = {}
SNRS = [20, 10, 5, 0]
NOISE_TEST_FRACTION = 0.3    # share of noise files kept for test / dev
DEV_CLEAN_PROB = 0.25        # dev: each utterance gets one condition
TRAIN_CLEAN_PROB = 0.25      # ft-noisy: share of clean samples
TRAIN_SNR_RANGE = (0.0, 20.0)

# Model and decoding (fixed for every experiment)
MODEL = "phowhisper-small"
MODEL_ID = "vinai/PhoWhisper-small"
GENERATION_CONFIG_FALLBACK = "openai/whisper-small"   # same tokenizer; used if the checkpoint lacks a language table
LANGUAGE = "vi"
NUM_BEAMS = 1                # greedy
MAX_NEW_TOKENS = 400
EVAL_BATCH = 16

# Fine-tuning
VARIANT = "ft-noisy" if NOISE_DIRS else "ft-clean"
EPOCHS = 5
TRAIN_BATCH = 16
LR = 2e-4
WARMUP_FRACTION = 0.1
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
LORA_TARGETS = ["q_proj", "v_proj"]
MAX_LABEL_TOKENS = 448
NUM_WORKERS = 2
SEED = 42

SR = 16000
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Code: clone and install

If the repo is private, add a Kaggle secret `GITHUB_TOKEN` (*Add-ons → Secrets*).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _token = None
_url = REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL

!rm -rf {CODE_DIR}
!git clone -q {_url} {CODE_DIR}
!git -C {CODE_DIR} checkout -q {COMMIT}
!git -C {CODE_DIR} log -1 --format="%h %ad %s" --date=short
!pip install -q -e "{CODE_DIR}[notebooks]" peft

In [ ]:
import sys
sys.path.insert(0, str(CODE_DIR))      # importable in this kernel without a restart

import functools
import gc
import io
import math
import re

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display
from tqdm.auto import tqdm

from core import convert, load_audio, mix
from dataprep import normalize_text, take_segment
from evaluation import PREDICTION_COLUMNS, bootstrap_ci, compare, score, summarize

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
torch.manual_seed(SEED)
print(DEVICE, torch.__version__)

## 2. Data

Every split becomes a table with one row per file to transcribe:
`utt_id, idx, condition, dataset, noise_type, snr_db, noise_path, offset_s, text`.
`text` is the normalized reference (`normalize_text` of `raw_transcription`). Noise choices
are drawn with `SEED` and saved, so the same table rebuilds the same audio.
After this step the code only sees `(rows, audios)`.

In [ ]:
!find /kaggle/input -maxdepth 5 -name dataset_dict.json 2>/dev/null

In [ ]:
from datasets import Audio as HFAudio, load_from_disk


def find_fleurs():
    if FLEURS_DIR is not None:
        return Path(FLEURS_DIR)
    hits = sorted(Path("/kaggle/input").glob(f"**/{LANG}/dataset_dict.json"))
    if not hits:
        raise FileNotFoundError(f"no {LANG}/dataset_dict.json under /kaggle/input: attach a FLEURS "
                                f"dataset or set FLEURS_DIR")
    return hits[0].parent


fleurs_dir = find_fleurs()
fleurs = load_from_disk(str(fleurs_dir))
print(fleurs_dir)
print(fleurs)
if isinstance(fleurs["test"].features.get("audio"), HFAudio):
    fleurs = fleurs.cast_column("audio", HFAudio(decode=False))   # we decode ourselves (no torchcodec)
TEXT_COLUMN = "raw_transcription" if "raw_transcription" in fleurs["test"].column_names else "transcription"


def decode(audio):
    # A FLEURS "audio" value -> 16 kHz mono float32 array.
    if audio.get("array") is not None:
        return convert(np.asarray(audio["array"], dtype=np.float32), audio["sampling_rate"])
    if audio.get("bytes"):
        x, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
    elif audio.get("path") and Path(audio["path"]).exists():
        x, sr = sf.read(audio["path"], dtype="float32")
    else:
        raise ValueError(f"audio has neither bytes nor a readable path: {audio.get('path')}")
    return convert(x, sr)


@functools.lru_cache(maxsize=256)
def speech(split, idx):
    return decode(fleurs[SPLITS[split]][int(idx)]["audio"])


x = speech("test", 0)
print(len(x) / SR, "s |", fleurs["test"][0][TEXT_COLUMN])
Audio(x, rate=SR)

In [ ]:
def utterances(split, limit=None):
    # One row per kept utterance: idx in the HF split, normalized text, duration when known.
    d = fleurs[SPLITS[split]]
    df = pd.DataFrame({"idx": np.arange(len(d)), "text_raw": d[TEXT_COLUMN]})
    if "num_samples" in d.column_names:
        df["duration_s"] = np.asarray(d["num_samples"]) / SR
        df = df[df["duration_s"] <= MAX_DURATION_S]
    df["text"] = df["text_raw"].map(normalize_text)
    df = df[df["text"].str.len() > 0]
    if limit is not None:
        df = df.sample(n=min(limit, len(df)), random_state=SEED).sort_values("idx")
    df["utt_id"] = [f"fleurs_{split}_{i:05d}" for i in df["idx"]]
    df["dataset"] = "fleurs"
    return df.reset_index(drop=True)


AUDIO_EXT = {".wav", ".flac", ".ogg", ".mp3"}


def noise_pools():
    # {noise_type: {"test": [paths], "train": [paths]}}, split by file with SEED.
    pools = {}
    for noise_type, folder in NOISE_DIRS.items():
        files = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in AUDIO_EXT)
        if not files:
            raise FileNotFoundError(f"no audio files under {folder}")
        order = np.random.default_rng(SEED).permutation(len(files))
        n_test = max(1, round(NOISE_TEST_FRACTION * len(files)))
        test = [files[i] for i in order[:n_test]]
        train = [files[i] for i in order[n_test:]] or test
        pools[noise_type] = {"test": test, "train": train}
        print(f"{noise_type}: {len(test)} test files, {len(train)} train files")
    return pools


@functools.lru_cache(maxsize=64)
def noise_audio(path):
    return load_audio(path)


def noise_duration_s(path):
    return sf.info(str(path)).duration


def with_noise(row, rng, pool, noise_type, snr_db):
    path = pool[noise_type]["test"][rng.integers(len(pool[noise_type]["test"]))]
    return {**row, "condition": f"{noise_type}_{snr_db:g}dB", "noise_type": noise_type,
            "snr_db": float(snr_db), "noise_path": str(path),
            "offset_s": round(float(rng.uniform(0, noise_duration_s(path))), 3)}


def clean_row(row):
    return {**row, "condition": "clean", "noise_type": "clean", "snr_db": np.nan,
            "noise_path": "", "offset_s": 0.0}


def build_test_set(utts, pool):
    # Every utterance × (clean + every noise type × every SNR).
    rng = np.random.default_rng(SEED)
    rows = []
    for row in utts.to_dict("records"):
        rows.append(clean_row(row))
        for noise_type in pool:
            for snr_db in SNRS:
                rows.append(with_noise(row, rng, pool, noise_type, snr_db))
    return pd.DataFrame(rows)


def build_dev_set(utts, pool):
    # One condition per utterance: clean with DEV_CLEAN_PROB, otherwise a random noise and SNR.
    rng = np.random.default_rng(SEED + 1)
    rows = []
    for row in utts.to_dict("records"):
        if not pool or rng.random() < DEV_CLEAN_PROB:
            rows.append(clean_row(row))
        else:
            noise_type = list(pool)[rng.integers(len(pool))]
            rows.append(with_noise(row, rng, pool, noise_type, SNRS[rng.integers(len(SNRS))]))
    return pd.DataFrame(rows)


EVAL_COLUMNS = ["utt_id", "split", "idx", "condition", "dataset", "noise_type", "snr_db",
                "noise_path", "offset_s", "text"]
pools = noise_pools()
test_set = build_test_set(utterances("test", LIMIT_TEST).assign(split="test"), pools)[EVAL_COLUMNS]
dev_set = build_dev_set(utterances("dev", LIMIT_DEV).assign(split="dev"), pools)[EVAL_COLUMNS]
test_set.to_csv(OUT / "test_set.csv", index=False)
dev_set.to_csv(OUT / "dev_set.csv", index=False)
print(f"test: {len(test_set)} files, {test_set['utt_id'].nunique()} utterances | dev: {len(dev_set)} files")
display(test_set["condition"].value_counts())
test_set.head(3)

In [ ]:
def render(row):
    # One table row -> the audio the model hears.
    x = speech(row["split"], row["idx"])
    if row["noise_type"] == "clean":
        return x
    noise = take_segment(noise_audio(row["noise_path"]), row["offset_s"], len(x))
    return mix(x, noise, row["snr_db"])


def iter_batches(table, batch_size):
    # Yield (rows, audios); rows of the same utterance stay together so speech() is cached.
    table = table.sort_values(["split", "idx"], kind="stable")
    for start in range(0, len(table), batch_size):
        rows = table.iloc[start:start + batch_size]
        yield rows, [render(r) for r in rows.to_dict("records")]


row = test_set.iloc[-1].to_dict()
print(row["condition"], "|", row["text"])
Audio(render(row), rate=SR)

## 3. Model and `transcribe`

Decoding is fixed for every experiment: Vietnamese, transcribe task, greedy
(`num_beams=1`, no sampling), at most `MAX_NEW_TOKENS` tokens. The output is returned as the
model writes it; scoring normalizes both sides.

In [ ]:
from transformers import GenerationConfig, WhisperForConditionalGeneration, WhisperProcessor

DECODE = dict(language=LANGUAGE, task="transcribe", num_beams=NUM_BEAMS, do_sample=False,
              max_new_tokens=MAX_NEW_TOKENS)

processor = WhisperProcessor.from_pretrained(MODEL_ID)


def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    if getattr(model.generation_config, "lang_to_id", None) is None:
        # Some fine-tuned checkpoints lack the language table that generate(language=...) needs.
        model.generation_config = GenerationConfig.from_pretrained(GENERATION_CONFIG_FALLBACK)
    model.generation_config.forced_decoder_ids = None
    model.config.forced_decoder_ids = None
    return model.to(DEVICE).eval()


@torch.no_grad()
def transcribe(model, processor, audios):
    # list of 16 kHz float32 arrays -> list of raw transcripts
    features = processor.feature_extractor(audios, sampling_rate=SR,
                                           return_tensors="pt").input_features.to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        ids = model.generate(input_features=features, **DECODE)
    return [text.strip() for text in processor.batch_decode(ids, skip_special_tokens=True)]


model = load_base_model()
print(transcribe(model, processor, [speech("test", 0)]))

### Shared helpers: run a model on a table, evaluate predictions

`evaluate` reuses `evaluation.score` (same `normalize_text` on reference and hypothesis) and
`summarize` (corpus-level WER / CER in percent, split into substitutions, deletions,
insertions).

In [ ]:
VI_DIACRITICS = re.compile("[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]",
                           re.IGNORECASE)
# Latin letters with Vietnamese diacritics, digits, common punctuation
ALLOWED = re.compile(r"[\s0-9A-Za-zÀ-ɏ̀-ͯḀ-ỿ.,!?;:'\"()%&/\-–…]")
SNR_ORDER = ["clean", "20 dB", "10 dB", "5 dB", "0 dB"]


def run_predictions(model, table, variant, batch_size=EVAL_BATCH, desc=None):
    rows, hyps = [], []
    for batch_rows, audios in tqdm(iter_batches(table, batch_size),
                                   total=math.ceil(len(table) / batch_size), desc=desc or variant):
        rows.append(batch_rows)
        hyps += transcribe(model, processor, audios)
    out = pd.concat(rows)[["utt_id", "condition", "dataset", "noise_type", "snr_db", "text"]]
    return out.assign(model=MODEL, variant=variant, hyp=hyps).reset_index(drop=True)


def predictions_path(variant):
    return OUT / f"predictions_{MODEL}_{variant}.csv"


def run_or_load(model, table, variant):
    # Reuse the saved CSV when it covers exactly this table; otherwise run and save.
    path = predictions_path(variant)
    if path.exists():
        saved = pd.read_csv(path, dtype={"hyp": str}, keep_default_na=False)
        if set(zip(saved["utt_id"], saved["condition"])) == set(zip(table["utt_id"], table["condition"])):
            print(f"loaded {path}")
            return saved
    pred = run_predictions(model, table, variant)
    pred.to_csv(path, index=False)
    print(f"wrote {path} ({len(pred)} rows)")
    return pred


def check_outputs(pred):
    report = pred[["utt_id", "condition", "text", "hyp"]].copy()
    report["diacritics"] = report["hyp"].str.contains(VI_DIACRITICS)
    report["odd_chars"] = report["hyp"].map(lambda t: "".join(sorted({c for c in t if not ALLOWED.match(c)})))
    report["special_token"] = report["hyp"].str.contains("<|", regex=False)
    report["empty"] = report["hyp"].str.strip().eq("")
    return report


def evaluate(pred, table, by=("model", "variant", "dataset", "noise_type", "snr_db")):
    plan = table[["utt_id", "condition", "dataset", "noise_type", "snr_db"]]
    refs = table[["utt_id", "text"]].drop_duplicates("utt_id")
    scored = score(pred[PREDICTION_COLUMNS], plan, refs)
    return scored, summarize(scored, by=list(by))


def wer_grid(summary, metric="wer"):
    # One row per system × noise type, one column per SNR.
    s = summary.assign(snr=summary["snr_db"].map(lambda v: "clean" if pd.isna(v) else f"{v:g} dB"))
    grid = s.pivot_table(index=["model", "variant", "dataset", "noise_type"], columns="snr",
                         values=metric, aggfunc="first")
    return grid[[c for c in SNR_ORDER if c in grid.columns]]


def worst(scored, n=5):
    errors = scored[["w_sub", "w_del", "w_ins"]].sum(axis=1) / scored["w_n_ref"].clip(lower=1)
    return scored.assign(utt_wer=(100 * errors).round(1)).nlargest(n, "utt_wer")[
        ["utt_id", "condition", "utt_wer", "ref", "hyp_norm"]]

## 4. Zero-shot test

Sanity check on 10 clean files (read every transcript), then the full test table.

In [ ]:
sample = test_set[test_set["condition"] == "clean"].head(10)
report = check_outputs(run_predictions(model, sample, "zero-shot", desc="sanity"))
for r in report.itertuples():
    print(f"[{r.utt_id}]\n  ref: {r.text}\n  hyp: {r.hyp}\n")
display(report[["utt_id", "diacritics", "odd_chars", "special_token", "empty"]])
problems = report[~report["diacritics"] | report["odd_chars"].ne("") | report["special_token"] | report["empty"]]
print("OK" if problems.empty else f"CHECK {len(problems)} transcripts above before the full run")

In [ ]:
pred_zs = run_or_load(model, test_set, "zero-shot")
pred_zs.sample(5, random_state=SEED)[["condition", "text", "hyp"]]

## 5. Evaluate

WER / CER per noise type × SNR, split into substitutions, deletions and insertions;
95% bootstrap intervals; the worst utterances.

In [ ]:
scored_zs, summary_zs = evaluate(pred_zs, test_set)
print("WER (%)"); display(wer_grid(summary_zs))
print("CER (%)"); display(wer_grid(summary_zs, "cer"))
display(summary_zs[["noise_type", "snr_db", "wer", "cer", "sub_pct", "del_pct", "ins_pct"]])
display(bootstrap_ci(scored_zs))
display(worst(scored_zs))

## 6. Fine-tuning setup

FLEURS train only. `ft-clean` serves clean audio; `ft-noisy` (when `NOISE_DIRS` is set)
mixes a train-pool noise file at a random SNR in `TRAIN_SNR_RANGE`, clean with
`TRAIN_CLEAN_PROB`. Draws depend on `(SEED, epoch, index)`: each epoch gets new noise and a
rerun gets the same samples. LoRA is added to `q_proj` / `v_proj` of the encoder and
decoder; the base weights are frozen.

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import get_linear_schedule_with_warmup


class FleursTrain(Dataset):
    def __init__(self, utts, pools, noisy):
        self.idx = utts["idx"].tolist()
        self.text = utts["text"].tolist()
        self.pools = pools if noisy else {}
        self.epoch = 1

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x = decode(fleurs[SPLITS["train"]][self.idx[i]]["audio"])
        if self.pools:
            rng = np.random.default_rng([SEED, self.epoch, i])
            if rng.random() >= TRAIN_CLEAN_PROB:
                noise_type = list(self.pools)[rng.integers(len(self.pools))]
                files = self.pools[noise_type]["train"]
                path = files[rng.integers(len(files))]
                noise = noise_audio(path)
                offset_s = float(rng.uniform(0, len(noise) / SR))
                x = mix(x, take_segment(noise, offset_s, len(x)), float(rng.uniform(*TRAIN_SNR_RANGE)))
        return x, self.text[i]


train_set = FleursTrain(utterances("train"), pools, noisy=VARIANT == "ft-noisy")
print(f"{VARIANT}: {len(train_set)} training utterances")

tokenizer = processor.tokenizer
tokenizer.set_prefix_tokens(language="vietnamese", task="transcribe", predict_timestamps=False)

for name in ["model", "ft_model", "best_model"]:          # free GPU memory from earlier steps
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

base = load_base_model()
DECODER_START = base.config.decoder_start_token_id
ft_model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                           target_modules=LORA_TARGETS, bias="none"))
ft_model.print_trainable_parameters()


def collate(items):
    audios, texts = zip(*items)
    features = processor.feature_extractor(list(audios), sampling_rate=SR,
                                           return_tensors="pt").input_features
    tokens = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LABEL_TOKENS,
                       return_tensors="pt")
    labels = tokens.input_ids.masked_fill(tokens.attention_mask.ne(1), -100)   # ignore padding
    if (labels[:, 0] == DECODER_START).all():
        labels = labels[:, 1:]             # the model prepends the start token itself
    return features, labels

In [ ]:
# Check one batch and listen to one training sample
train_set.set_epoch(1)
audio, text = train_set[0]
print(text)
display(Audio(audio, rate=SR))

features, labels = collate([train_set[i] for i in range(4)])
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
    loss = ft_model(input_features=features.to(DEVICE), labels=labels.to(DEVICE)).loss
print(features.shape, labels.shape, f"loss {loss.item():.3f}")
print(tokenizer.decode(labels[0][labels[0] != -100]))

## 7. Fine-tuning

Dev WER is measured before training (epoch 0 = zero-shot) and after each epoch. The adapter
with the lowest dev WER is saved. The test set is not used here.

In [ ]:
def dev_wer(model):
    pred = run_predictions(model, dev_set, VARIANT, desc="dev")
    _, summary = evaluate(pred, dev_set, by=("model", "variant"))
    return float(summary["wer"].iloc[0])


ADAPTER_DIR = OUT / f"lora_{MODEL}_{VARIANT}"
steps_per_epoch = math.ceil(len(train_set) / TRAIN_BATCH)
optimizer = torch.optim.AdamW([p for p in ft_model.parameters() if p.requires_grad], lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * EPOCHS * steps_per_epoch),
                                            EPOCHS * steps_per_epoch)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

ft_model.eval()
history = [{"epoch": 0, "train_loss": None, "dev_wer": dev_wer(ft_model)}]
print(history[-1])
best_wer = math.inf

for epoch in range(1, EPOCHS + 1):
    train_set.set_epoch(epoch)             # new noise draws for this epoch
    loader = DataLoader(train_set, batch_size=TRAIN_BATCH, shuffle=True, collate_fn=collate,
                        num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(SEED + epoch))
    ft_model.train()
    losses = []
    for features, labels in tqdm(loader, desc=f"epoch {epoch}"):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            loss = ft_model(input_features=features.to(DEVICE), labels=labels.to(DEVICE)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        losses.append(loss.item())
    ft_model.eval()
    wer = dev_wer(ft_model)
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "dev_wer": wer})
    if wer < best_wer:
        best_wer = wer
        ft_model.save_pretrained(ADAPTER_DIR)
    print(history[-1], "(best)" if wer == best_wer else "")
    pd.DataFrame(history).to_csv(OUT / f"history_{MODEL}_{VARIANT}.csv", index=False)

pd.DataFrame(history)

## 8. Test the fine-tuned model

Loads the best adapter, runs the same test table with the same decoding, and compares it
with zero-shot. `compare` is a paired bootstrap of `WER(fine-tuned) − WER(zero-shot)`: a
negative `diff` means fewer errors; `significant` is True when the 95% interval excludes 0.

In [ ]:
for name in ["model", "ft_model", "base"]:
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

best_model = PeftModel.from_pretrained(load_base_model(), ADAPTER_DIR).to(DEVICE).eval()
pred_ft = run_or_load(best_model, test_set, VARIANT)
pred_zs = run_or_load(load_base_model() if not predictions_path("zero-shot").exists() else None,
                      test_set, "zero-shot")
scored_ft, _ = evaluate(pred_ft, test_set)
scored_zs, _ = evaluate(pred_zs, test_set)

both = pd.concat([scored_zs, scored_ft], ignore_index=True)
print("WER (%)"); display(wer_grid(summarize(both)))
print("CER (%)"); display(wer_grid(summarize(both), "cer"))
display(compare(both, (MODEL, "zero-shot"), (MODEL, VARIANT)))
display(worst(scored_ft))